# Peak Detection for Pred vs. Ground Truth

This notebook extracts peaks from a predicted boundary-probability series and the ground-truth series, and writes CSVs with an `is_peak` flag.

In [1]:
from pathlib import Path
import subprocess
import sys

# Adjust repo_root if needed
base_dir = Path(".").resolve()
repo_root = base_dir.parent.parent
# repo_root = Path("/Users/toddywang/Documents/VsCodeProjects/xmltoexp")

cfg_path = repo_root / "MERIX SUBMISSION" / "MIREX_Model" / "config_beat_mazurka.yaml"
model_root = repo_root / "MERIX SUBMISSION" / "MIREX_Model" / "check" / "beat_mazurka"
ckpts = sorted(
    [p for p in model_root.glob("beat_mazurka_*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
)
if not ckpts:
    raise FileNotFoundError(f"No checkpoints found under {model_root}")
model_path = ckpts[-1] / "best.pt"

input_npz = repo_root / "MERIX SUBMISSION" / "MIREX_Model" / "beat_data_mazurka" / "M06-3.npz"
output_csv = repo_root / "MERIX SUBMISSION" / "MIREX_Model" / "out" / "pred_M06-3.csv"

# For dual-head models, set head = "prob" or "dist".
# For single-head models, leave as None.
head = None

cmd = [
    sys.executable,
    str(repo_root / "MERIX SUBMISSION" / "MIREX_Model" / "infer_beat.py"),
    "--config",
    str(cfg_path),
    "--model_path",
    str(model_path),
    "--input_npz",
    str(input_npz),
    "--output_csv",
    str(output_csv),
]
if head:
    cmd += ["--head", head]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: /Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/.venv/bin/python /Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/infer_beat.py --config /Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/config_beat_mazurka.yaml --model_path /Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/check/beat_mazurka/beat_mazurka_20260118_223507/best.pt --input_npz /Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/beat_data_mazurka/M06-3.npz --output_csv /Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/out/pred_M06-3.csv


/Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Saved predictions to /Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/out/pred_M06-3.csv | beats=294


CompletedProcess(args=['/Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/.venv/bin/python', '/Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/infer_beat.py', '--config', '/Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/config_beat_mazurka.yaml', '--model_path', '/Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/check/beat_mazurka/beat_mazurka_20260118_223507/best.pt', '--input_npz', '/Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/beat_data_mazurka/M06-3.npz', '--output_csv', '/Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/out/pred_M06-3.csv'], returncode=0)

In [2]:
from pathlib import Path
import csv
import numpy as np


def read_series(path: Path):
    idx = []
    vals = []
    with path.open() as f:
        r = csv.reader(f)
        header = next(r)
        for i, p in r:
            idx.append(int(i))
            vals.append(float(p))
    return np.asarray(idx, dtype=int), np.asarray(vals, dtype=float)


def peak_pick(y, min_dist=6, height=None, prominence=None):
    try:
        from scipy.signal import find_peaks
        kwargs = {"distance": int(min_dist)}
        if height is not None:
            kwargs["height"] = float(height)
        if prominence is not None:
            kwargs["prominence"] = float(prominence)
        peaks, _ = find_peaks(y, **kwargs)
        return peaks.astype(int)
    except Exception:
        peaks = []
        last = -int(min_dist) - 1
        for i in range(1, len(y) - 1):
            if y[i] >= y[i - 1] and y[i] >= y[i + 1]:
                if height is None or y[i] >= height:
                    if i - last >= min_dist:
                        peaks.append(i)
                        last = i
        return np.asarray(peaks, dtype=int)


def write_peaks(path: Path, idx, y, peaks):
    peak_set = set(peaks.tolist())
    with path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["beat_index", "boundary_probability", "is_peak"])
        for i, p in zip(idx, y):
            w.writerow([int(i), float(p), 1 if int(i) in peak_set else 0])
    print("saved", path)
    print("peaks", peaks.tolist())


In [3]:
# Adjust repo_root if needed
base_dir = Path(".").resolve()
repo_root = base_dir.parent.parent
# repo_root = Path("/Users/toddywang/Documents/VsCodeProjects/xmltoexp")

pred_path = repo_root / "MERIX SUBMISSION" / "MIREX_Model" / "out" / "pred_M06-3.csv"
true_path = repo_root / "out" / "mazurka_boundary_probs" / "M06-3_boundary_prob.csv"

pred_out = repo_root / "MERIX SUBMISSION" / "MIREX_Model" / "out" / "pred_M06-3_peaks.csv"
true_out = repo_root / "out" / "mazurka_boundary_probs" / "M06-3_boundary_prob_peaks.csv"

# Peak parameters
min_dist = 6
pred_height = 0.3
pred_prominence = 0.05
true_height = 0.1
true_prominence = 0.05

# Predicted peaks
idx, y = read_series(pred_path)
peaks = peak_pick(y, min_dist=min_dist, height=pred_height, prominence=pred_prominence)
write_peaks(pred_out, idx, y, peaks)

# Ground-truth peaks
idx_t, y_t = read_series(true_path)
peaks_t = peak_pick(y_t, min_dist=min_dist, height=true_height, prominence=true_prominence)
write_peaks(true_out, idx_t, y_t, peaks_t)


saved /Users/toddywang/Documents/VsCodeProjects/xmltoexp/MERIX SUBMISSION/MIREX_Model/out/pred_M06-3_peaks.csv
peaks [9, 24, 35, 41, 49, 59, 65, 71, 85, 98, 110, 120, 128, 140, 159, 173, 192, 219, 225, 231, 239, 245, 255, 263, 270]
saved /Users/toddywang/Documents/VsCodeProjects/xmltoexp/out/mazurka_boundary_probs/M06-3_boundary_prob_peaks.csv
peaks [11, 24, 35, 47, 61, 71, 83, 96, 113, 124, 134, 144, 156, 162, 169, 180, 193, 205, 215, 228, 239, 251, 264, 281]
